In [2]:
# Day 9 - Processed E-commerce Dataset
# loading orders, customers, products and merging them into one clean dataset

import pandas as pd
import numpy as np

orders = pd.read_csv('Day9_Orders.csv')
customers = pd.read_csv('Day9_Customers.csv')
products = pd.read_csv('Day9_Products.csv')

print(orders.shape, customers.shape, products.shape)
orders.head()

(120, 7) (30, 5) (20, 5)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


In [3]:
# concat() demo - splitting orders in half and joining them back to show how concat works
orders_part1 = orders.iloc[:60]
orders_part2 = orders.iloc[60:]

orders_combined = pd.concat([orders_part1, orders_part2], axis=0, ignore_index=True)
print(orders_combined.shape)
orders_combined.equals(orders)  # should be True

(120, 7)


True

In [4]:
# merge() - bringing customer and product details into the orders table
orders_customers = pd.merge(orders, customers, on='Customer_ID', how='left')
full_data = pd.merge(orders_customers, products, on='Product_ID', how='left')

full_data.isna().sum()  # check nothing broke during the merge

,0
Order_ID,0
Order_Date,0
Customer_ID,0
Product_ID,0
Quantity,0
Payment_Method,0
Order_Status,0
Customer_Name,0
City,0
Region,0


In [5]:
# apply() - creating new columns
full_data['Total_Amount'] = full_data.apply(lambda row: row['Unit_Price'] * row['Quantity'], axis=1)

def get_order_size(qty):
    if qty == 1:
        return 'Small'
    elif qty in (2, 3):
        return 'Medium'
    else:
        return 'Large'

full_data['Order_Size'] = full_data['Quantity'].apply(get_order_size)
full_data['Customer_Name'] = full_data['Customer_Name'].apply(lambda name: name.title())

In [6]:
# datetime operations - extracting month, day, weekday from the order date
full_data['Order_Date'] = pd.to_datetime(full_data['Order_Date'])
full_data['Order_Month'] = full_data['Order_Date'].dt.month_name()
full_data['Order_Day'] = full_data['Order_Date'].dt.day
full_data['Order_Weekday'] = full_data['Order_Date'].dt.day_name()

In [7]:
# final cleaned dataset - picking useful columns and sorting by date
final_df = full_data[[
    'Order_ID', 'Order_Date', 'Order_Month', 'Order_Day', 'Order_Weekday',
    'Customer_ID', 'Customer_Name', 'City', 'Region', 'Membership_Type',
    'Product_ID', 'Product_Name', 'Category', 'Brand',
    'Quantity', 'Unit_Price', 'Total_Amount', 'Order_Size',
    'Payment_Method', 'Order_Status'
]].copy()

final_df = final_df.sort_values('Order_Date').reset_index(drop=True)
final_df.head(10)

,Order_ID,Order_Date,Order_Month,Order_Day,Order_Weekday,Customer_ID,Customer_Name,City,Region,Membership_Type,Product_ID,Product_Name,Category,Brand,Quantity,Unit_Price,Total_Amount,Order_Size,Payment_Method,Order_Status
0,O0051,2026-01-01,January,1,Thursday,C008,Sara Ahmed,Hyderabad,South,Premium,P017,Yoga Mat,Sports,FitLife,4,899,3596,Large,Net Banking,Delivered
1,O0091,2026-01-01,January,1,Thursday,C002,Zoya Khan,Delhi,North,Regular,P011,Air Fryer,Home & Kitchen,CookSmart,2,4999,9998,Medium,Credit Card,Delivered
2,O0022,2026-01-02,January,2,Friday,C023,Nikhil Sood,Chandigarh,North,Premium,P014,Data Science Handbook,Books,DataPress,4,899,3596,Large,Net Banking,Delivered
3,O0106,2026-01-02,January,2,Friday,C001,Aarav Sharma,Srinagar,North,Premium,P009,Coffee Maker,Home & Kitchen,HomeBrew,1,3499,3499,Small,Net Banking,Delivered
4,O0076,2026-01-03,January,3,Saturday,C019,Yusuf Dar,Srinagar,North,New,P015,Machine Learning Basics,Books,AIPress,1,999,999,Small,Debit Card,Delivered
5,O0108,2026-01-04,January,4,Sunday,C006,Ishita Gupta,Bengaluru,South,Premium,P020,Dumbbell Set,Sports,StrongFit,1,1999,1999,Small,Net Banking,Delivered
6,O0041,2026-01-04,January,4,Sunday,C008,Sara Ahmed,Hyderabad,South,Premium,P012,Water Bottle,Home & Kitchen,HydroLife,5,699,3495,Large,Cash on Delivery,Delivered
7,O0018,2026-01-05,January,5,Monday,C003,Rohan Mehta,Mumbai,West,Premium,P011,Air Fryer,Home & Kitchen,CookSmart,4,4999,19996,Large,Net Banking,Cancelled
8,O0104,2026-01-05,January,5,Monday,C011,Vivaan Kapoor,Jaipur,North,New,P006,Hoodie,Clothing,UrbanWear,3,1599,4797,Medium,Debit Card,Shipped
9,O0110,2026-01-05,January,5,Monday,C007,Aditya Verma,Pune,West,Regular,P005,Power Bank,Electronics,VoltPlus,1,1199,1199,Small,UPI,Delivered


In [8]:
# export final processed dataset
final_df.to_csv('Processed_Ecommerce_Dataset.csv', index=False)
print('saved! rows:', len(final_df))

saved! rows: 120
